### **Import libraries and Data**


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [2]:
loan_data = pd.read_csv("/mnt/Docs/Git_Repo/Credit_Scorecard/Data/loan_data_v1.csv")
loan_data.head()

,age,gender,education_level,income,work_experience,home_ownership_status,loan_intent,loan_to_income_ratio,credit_history_length,previous_loan_defaults_on_file,loan_status
0,22,female,Master,71948,0,RENT,PERSONAL,0.49,3,No,1
1,21,female,High School,12282,0,OWN,EDUCATION,0.08,2,Yes,0
2,25,female,High School,12438,3,MORTGAGE,MEDICAL,0.44,3,No,1
3,23,female,Bachelor,79753,0,RENT,MEDICAL,0.44,2,No,1
4,24,male,Master,66135,1,RENT,MEDICAL,0.53,4,No,1


### **Transforming Numerical Variables to Categorical to Calculate WOE**

In [3]:
num_cols = loan_data.drop(columns = ['loan_status']).select_dtypes(include=np.number).columns.tolist()
print(num_cols)

['age', 'income', 'work_experience', 'loan_to_income_ratio', 'credit_history_length']


#### **AGE**

In [4]:
age_bin = [20,35,45,65]
age_labels = ["young_adults","prime_adults","middle_aged"]
loan_data['age_bin'] = pd.cut(loan_data['age'],bins=age_bin,labels=age_labels,include_lowest=True)

In [5]:
loan_data['age_bin'].value_counts()

age_bin
young_adults    40514
prime_adults     3724
middle_aged       697
Name: count, dtype: int64

#### __Income__

In [6]:
# Let's change the income values to '000s for better readability
loan_data['income'] = round(loan_data['income']/1e3,2)
loan_data['income'].describe()

count    44935.000000
mean        79.853324
std         62.579880
min          8.000000
25%         47.175000
50%         67.040000
75%         95.755000
max       2448.660000
Name: income, dtype: float64

In [7]:
income_bins = [0,40,60,120,180,np.inf]
income_labels = ['low','lower-middle','middle','upper-middle','high']
loan_data['income_binned'] = pd.cut(loan_data['income'], bins=income_bins,labels=income_labels,
                                   include_lowest=True)
loan_data['income_binned'].value_counts(normalize=True)

income_binned
middle          0.452543
lower-middle    0.242595
low             0.163792
upper-middle    0.100879
high            0.040191
Name: proportion, dtype: float64

In [8]:
loan_data["work_experience"].describe()

count    44935.000000
mean         5.342272
std          5.745546
min          0.000000
25%          1.000000
50%          4.000000
75%          8.000000
max         40.000000
Name: work_experience, dtype: float64

#### __Work Experience__

In [9]:
"""
We will bin the 'work experience' variable based on the risk profile as well. 
For example, employees with less than 2 years of experience may be considered higher risk, while those with more than 10 years of experience may be considered lower risk.
"""
experience_bins = [0, 3, 7, 12, 20, np.inf]
experience_labels = [
    "entry_level",   # 0-3
    "early_career",   # 2-7
    "mid_level",      # 7-12
    "established",    # 12-20
    "veteran"         # 20+
]
loan_data["work_experience_binned"] = pd.cut(loan_data["work_experience"], bins=experience_bins, labels=experience_labels, include_lowest=True)
loan_data['work_experience_binned'].value_counts(normalize=True)


work_experience_binned
entry_level     0.481829
early_career    0.254701
mid_level       0.154045
established     0.083699
veteran         0.025726
Name: proportion, dtype: float64

#### __Loan_to_Income_Ratio__

In [10]:
loan_data['loan_to_income_ratio'].describe()

count    44935.000000
mean         0.139724
std          0.087199
min          0.000000
25%          0.070000
50%          0.120000
75%          0.190000
max          0.660000
Name: loan_to_income_ratio, dtype: float64

In [13]:
loan_income_ratio_bin = [0,0.25,0.45,1]
loan_income_ratio_labels = ["low","medium","high"]

loan_data['loan_income_ratio_bin'] = pd.cut(loan_data['loan_to_income_ratio'],bins=loan_income_ratio_bin,
                                           labels=loan_income_ratio_labels,include_lowest=True)

loan_data['loan_income_ratio_bin'].value_counts(normalize=True)

loan_income_ratio_bin
low       0.888239
medium    0.108690
high      0.003071
Name: proportion, dtype: float64

#### __Credit History Length__

In [14]:
loan_data['credit_history_length'].describe()

count    44935.000000
mean         5.842773
std          3.820994
min          2.000000
25%          3.000000
50%          4.000000
75%          8.000000
max         30.000000
Name: credit_history_length, dtype: float64

In [15]:
cr_hist_bin = [2,5,10,np.inf]
cr_hist_label = ["2-5","5-10","10+"]

loan_data['cr_hist_bin'] = pd.cut(loan_data['credit_history_length'],bins=cr_hist_bin,
                                  labels=cr_hist_label,include_lowest=True) 
loan_data['cr_hist_bin'].value_counts(normalize=True)

cr_hist_bin
2-5     0.591521
5-10    0.307044
10+     0.101435
Name: proportion, dtype: float64